# Statistical Pattern Recognition - Computer Assignment #4

## Audited MLP with Online Backpropagation

This notebook is a clean, reproducible implementation of the supplied assignment. It follows the required 3-input (x1, x2, +1 bias) -> 4-hidden-sigmoid (+1 bias) -> 2-output-sigmoid architecture, uses the instructor-provided initial weights, processes the supplied training order without shuffling, trains for exactly **500 epochs** with learning rate **0.2**, classifies by the larger output node, and generates the required trained weights, confusion matrix, and misclassified-sample list.

The supplied workbook is preserved as the authoritative dataset. Because no verifiable public host for the exact instructor workbook was found, the notebook includes a self-contained copy rather than substituting an unrelated dataset or inventing a URL. An optional `ASSIGNMENT_DATA_URL` environment variable can be used when a legitimate hosted copy exists.

In [1]:
# Install only packages that are missing in the current environment.
import importlib.util
import subprocess
import sys

required_packages = {
    "openpyxl": "openpyxl",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "numpy": "numpy",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from pathlib import Path
import base64
import os
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openpyxl import load_workbook

OUTPUT_DIR = Path("assignment_4_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LEARNING_RATE = 0.2
EPOCHS = 500
TARGET_VALUES = {(0.95, 0.05), (0.05, 0.95)}

from IPython.display import display


In [2]:
DATASET_B64 = "..."

def _embedded_dataset_bytes() -> bytes:
    """Decode the embedded assignment dataset workbook."""
    return base64.b64decode(DATASET_B64)

def prepare_dataset(dataset_path: str | Path, output_dir: Path) -> Path:
    """Resolve the dataset from a URL, an existing local file, or the embedded copy."""
    dataset_path = Path(dataset_path)
    url = os.environ.get("ASSIGNMENT_DATA_URL", "").strip()

    if url:
        try:
            from urllib.request import urlopen

            with urlopen(url, timeout=30) as response:
                dataset_path.write_bytes(response.read())
            return dataset_path
        except Exception as exc:
            print(f"Dataset URL retrieval failed; using the supplied bundled dataset instead: {exc}")

    if dataset_path.exists():
        return dataset_path

    dataset_path.parent.mkdir(parents=True, exist_ok=True)
    dataset_path.write_bytes(_embedded_dataset_bytes())
    return dataset_path


In [3]:
def load_assignment_data(dataset_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    """Load training data, testing data, and the two initial weight matrices."""
    workbook = load_workbook(dataset_path, data_only=True, read_only=True)
    expected_sheets = {"Training Set", "Testing Set", "Initial Weights"}
    missing = expected_sheets.difference(workbook.sheetnames)
    if missing:
        raise ValueError(f"Dataset workbook is missing required sheets: {sorted(missing)}")

    train_ws = workbook["Training Set"]
    train_rows = list(train_ws.iter_rows(values_only=True))
    if train_rows and list(train_rows[0])[:5] == ["Class #", "x1", "x2", "Target 1", "Target 2"]:
        training_set = pd.DataFrame(train_rows[1:], columns=train_rows[0]).dropna(how="all")
    else:
        training_set = pd.DataFrame(train_rows[2:], columns=train_rows[1]).dropna(how="all")

    test_ws = workbook["Testing Set"]
    test_rows = list(test_ws.iter_rows(values_only=True))
    if test_rows and list(test_rows[0])[:3] == ["Class #", "x1", "x2"]:
        testing_set = pd.DataFrame(test_rows[1:], columns=test_rows[0]).dropna(how="all")
    else:
        testing_set = pd.DataFrame(test_rows[2:], columns=test_rows[1]).dropna(how="all")

    weight_ws = workbook["Initial Weights"]
    w = list(weight_ws.iter_rows(values_only=True))
    w = [list(row) for row in w]

    input_hidden = np.array(
        [
            [w[r][2] for r in range(4, 8)],
            [w[r][2] for r in range(8, 12)],
            [w[r][2] for r in range(12, 16)],
        ],
        dtype=float,
    )
    hidden_output = np.array(
        [
            [w[r][6], w[r + 1][6]]
            for r in range(4, 12, 2)
        ]
        + [[w[12][6], w[13][6]]],
        dtype=float,
    )
    workbook.close()

    return training_set, testing_set, input_hidden, hidden_output

def validate_assignment_data(
    training_set: pd.DataFrame,
    testing_set: pd.DataFrame,
    input_hidden: np.ndarray,
    hidden_output: np.ndarray,
) -> None:
    """Verify the structural assumptions required by the assignment."""
    required_train = ["Class #", "x1", "x2", "Target 1", "Target 2"]
    required_test = ["Class #", "x1", "x2"]
    if list(training_set.columns) != required_train:
        raise ValueError(f"Unexpected training columns: {training_set.columns.tolist()}")
    if list(testing_set.columns) != required_test:
        raise ValueError(f"Unexpected testing columns: {testing_set.columns.tolist()}")

    if training_set.shape != (1000, 5):
        raise ValueError(f"Expected 1000 training samples; found {training_set.shape[0]}")
    if testing_set.shape != (1000, 3):
        raise ValueError(f"Expected 1000 testing samples; found {testing_set.shape[0]}")

    train_classes = training_set["Class #"].astype(int).to_numpy()
    test_classes = testing_set["Class #"].astype(int).to_numpy()
    if set(train_classes) != {1, 2} or set(test_classes) != {1, 2}:
        raise ValueError("The assignment requires exactly two classes: 1 and 2.")

    expected_alternating = np.array([1 if i % 2 == 0 else 2 for i in range(len(train_classes))])
    if not np.array_equal(train_classes, expected_alternating):
        raise ValueError("Training samples are not in the required alternating class order.")

    features = training_set[["x1", "x2"]].to_numpy(dtype=float)
    if np.nanmin(features) < 0.2 or np.nanmax(features) > 0.8:
        raise ValueError("Training features must remain within the assignment normalization range [0.2, 0.8].")

    targets = training_set[["Target 1", "Target 2"]].to_numpy(dtype=float)
    unique_targets = {tuple(np.round(row, 8)) for row in targets}
    if unique_targets != TARGET_VALUES:
        raise ValueError(f"Unexpected target vectors: {sorted(unique_targets)}")

    if input_hidden.shape != (3, 4):
        raise ValueError(f"Input-to-hidden weights must have shape (3, 4), got {input_hidden.shape}.")
    if hidden_output.shape != (5, 2):
        raise ValueError(f"Hidden-to-output weights must have shape (5, 2), got {hidden_output.shape}.")

dataset_path = prepare_dataset(OUTPUT_DIR / "normal_assignment_4_dataset.xlsx", OUTPUT_DIR)
training_set, testing_set, initial_input_hidden, initial_hidden_output = load_assignment_data(dataset_path)
validate_assignment_data(training_set, testing_set, initial_input_hidden, initial_hidden_output)

print(f"Training samples: {len(training_set)}")
print(f"Testing samples: {len(testing_set)}")
print("Initial input-to-hidden shape:", initial_input_hidden.shape)
print("Initial hidden-to-output shape:", initial_hidden_output.shape)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)


Training samples: 1000
Testing samples: 1000
Initial input-to-hidden shape: (3, 4)
Initial hidden-to-output shape: (5, 2)
Learning rate: 0.2
Epochs: 500


In [4]:
def sigmoid(x: np.ndarray | float) -> np.ndarray:
    """Compute the sigmoid function with numerical overflow protection."""
    x = np.asarray(x, dtype=float)
    positive = x >= 0
    result = np.empty_like(x, dtype=float)
    result[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    result[~positive] = exp_x / (1.0 + exp_x)
    return result

def forward_pass(
    features: np.ndarray,
    input_hidden_weights: np.ndarray,
    hidden_output_weights: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Perform a complete forward pass through the assignment MLP."""
    input_with_bias = np.append(features, 1.0)
    hidden_inputs = input_with_bias @ input_hidden_weights
    hidden_outputs = sigmoid(hidden_inputs)

    hidden_with_bias = np.append(hidden_outputs, 1.0)
    output_inputs = hidden_with_bias @ hidden_output_weights
    final_outputs = sigmoid(output_inputs)
    return hidden_outputs, final_outputs

# Confirm a single forward pass before training.
example_hidden, example_output = forward_pass(
    training_set[["x1", "x2"]].iloc[0].to_numpy(dtype=float),
    initial_input_hidden,
    initial_hidden_output,
)
print("Example hidden output:", example_hidden)
print("Example final output:", example_output)


Example hidden output: [0.45893194 0.60448302 0.36255838 0.41847576]
Example final output: [0.5444438  0.59991286]


In [5]:
def forward_dataset(
    features: np.ndarray,
    input_hidden_weights: np.ndarray,
    hidden_output_weights: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Run a batch of samples through the network without changing weights."""
    input_with_bias = np.concatenate([features, np.ones((len(features), 1))], axis=1)
    hidden_outputs = sigmoid(input_with_bias @ input_hidden_weights)
    hidden_with_bias = np.concatenate([hidden_outputs, np.ones((len(hidden_outputs), 1))], axis=1)
    final_outputs = sigmoid(hidden_with_bias @ hidden_output_weights)
    return hidden_outputs, final_outputs

def train_mlp(
    features: np.ndarray,
    targets: np.ndarray,
    initial_input_hidden: np.ndarray,
    initial_hidden_output: np.ndarray,
    learning_rate: float = LEARNING_RATE,
    epochs: int = EPOCHS,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Train the network using online backpropagation for exactly 500 epochs."""
    input_hidden_weights = initial_input_hidden.astype(float, copy=True)
    hidden_output_weights = initial_hidden_output.astype(float, copy=True)
    history = []

    for epoch in range(1, epochs + 1):
        for features_i, target_i in zip(features, targets):
            hidden_outputs, final_outputs = forward_pass(
                features_i, input_hidden_weights, hidden_output_weights
            )

            output_error = target_i - final_outputs
            output_delta = output_error * final_outputs * (1.0 - final_outputs)

            # Compute the hidden-layer error with the output weights before
            # updating that layer, matching the standard online BP update.
            hidden_error = output_delta @ hidden_output_weights[:-1, :].T
            hidden_delta = hidden_error * hidden_outputs * (1.0 - hidden_outputs)

            hidden_output_weights[:-1, :] += learning_rate * np.outer(hidden_outputs, output_delta)
            hidden_output_weights[-1, :] += learning_rate * output_delta
            input_hidden_weights[:-1, :] += learning_rate * np.outer(features_i, hidden_delta)
            input_hidden_weights[-1, :] += learning_rate * hidden_delta

        _, epoch_outputs = forward_dataset(features, input_hidden_weights, hidden_output_weights)
        epoch_mse = float(np.mean((targets - epoch_outputs) ** 2))
        history.append({"epoch": epoch, "mean_squared_error": epoch_mse})

    return input_hidden_weights, hidden_output_weights, pd.DataFrame(history)

X_train = training_set[["x1", "x2"]].to_numpy(dtype=float)
y_train = training_set[["Target 1", "Target 2"]].to_numpy(dtype=float)
X_test = testing_set[["x1", "x2"]].to_numpy(dtype=float)

trained_input_hidden, trained_hidden_output, history = train_mlp(
    X_train,
    y_train,
    initial_input_hidden,
    initial_hidden_output,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
)

print("Training complete.")
print("Final training MSE:", float(history.iloc[-1]["mean_squared_error"]))


Training complete.
Final training MSE: 0.045295863792794246


In [6]:
def predict(
    features: np.ndarray,
    input_hidden_weights: np.ndarray,
    hidden_output_weights: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Return predicted classes and raw two-node output activations."""
    _, outputs = forward_dataset(features, input_hidden_weights, hidden_output_weights)
    predictions = np.argmax(outputs, axis=1) + 1
    return predictions.astype(int), outputs

predictions, raw_outputs = predict(X_test, trained_input_hidden, trained_hidden_output)
true_labels = testing_set["Class #"].astype(int).to_numpy()
accuracy = float(np.mean(predictions == true_labels))

confusion = np.array(
    [
        [np.sum((true_labels == 1) & (predictions == 1)), np.sum((true_labels == 1) & (predictions == 2))],
        [np.sum((true_labels == 2) & (predictions == 1)), np.sum((true_labels == 2) & (predictions == 2))],
    ],
    dtype=int,
)

mis_mask = predictions != true_labels
misclassified_count = int(np.sum(mis_mask))

print(f"Test accuracy: {accuracy:.4%}")
print(f"Misclassified samples: {misclassified_count}")
print("Confusion matrix:\n", confusion)


Test accuracy: 93.2000%
Misclassified samples: 68
Confusion matrix:
 [[465  35]
 [ 33 467]]


In [7]:
def save_trained_weights(path, input_hidden_weights, hidden_output_weights, learning_rate, epochs):
    """Save trained weights in the same logical layout as the assignment workbook."""
    rows = [
        ["Initial Values of the Weights", "", "", "", "", "", ""],
        ["", "", "", "", "", "", ""],
        ["Input to Hidden Weights", "", "", "", "Hidden to Output Weights", "", ""],
        ["From Input", "To Hidden", "", "", "From Hidden", "To Output", ""],
    ]
    for hidden_idx in range(4):
        rows.append([1, hidden_idx + 1, input_hidden_weights[0, hidden_idx], "", hidden_idx + 1, 1, hidden_output_weights[hidden_idx, 0]])
        rows.append([1, hidden_idx + 1, input_hidden_weights[0, hidden_idx], "", hidden_idx + 1, 2, hidden_output_weights[hidden_idx, 1]])
    for hidden_idx in range(4):
        rows[4 + hidden_idx] = [1, hidden_idx + 1, input_hidden_weights[0, hidden_idx], "", hidden_idx + 1, 1, hidden_output_weights[hidden_idx, 0]]
        rows[8 + hidden_idx] = [2, hidden_idx + 1, input_hidden_weights[1, hidden_idx], "", hidden_idx + 1, 2, hidden_output_weights[hidden_idx, 1]]
    rows.extend([
        ["Bias Node (+1)", 1, input_hidden_weights[2, 0], "", "Bias Node (+1)", 1, hidden_output_weights[4, 0]],
        ["Bias Node (+1)", 2, input_hidden_weights[2, 1], "", "Bias Node (+1)", 2, hidden_output_weights[4, 1]],
        ["Bias Node (+1)", 3, input_hidden_weights[2, 2], "", "", "", ""],
        ["Bias Node (+1)", 4, input_hidden_weights[2, 3], "", "", "", ""],
        ["", "", "", "", "", "", ""],
        ["", "", "", "", "", "", ""],
        ["Learning Rate", "", learning_rate, "", "", "", ""],
        ["# Iterations", "", epochs, "", "", "", ""],
    ])
    pd.DataFrame(rows).to_csv(path, index=False, header=False)


def save_weight_matrices(path, input_hidden_weights, hidden_output_weights):
    rows = []
    for i in range(3):
        source = "Bias Node (+1)" if i == 2 else f"Input {i + 1}"
        for j in range(4):
            rows.append({"layer": "input_to_hidden", "source": source, "target": f"Hidden {j + 1}", "weight": input_hidden_weights[i, j]})
    for i in range(5):
        source = "Bias Node (+1)" if i == 4 else f"Hidden {i + 1}"
        for j in range(2):
            rows.append({"layer": "hidden_to_output", "source": source, "target": f"Output {j + 1}", "weight": hidden_output_weights[i, j]})
    pd.DataFrame(rows).to_csv(path, index=False)


# Required statistics and machine-readable result files.
true_labels = testing_set["Class #"].astype(int).to_numpy()
mis_mask = predictions != true_labels

save_trained_weights(
    OUTPUT_DIR / "trained_weights.csv",
    trained_input_hidden,
    trained_hidden_output,
    LEARNING_RATE,
    EPOCHS,
)
save_weight_matrices(OUTPUT_DIR / "weight_matrices.csv", trained_input_hidden, trained_hidden_output)
pd.DataFrame(confusion, index=["Actual Class 1", "Actual Class 2"], columns=["Predicted Class 1", "Predicted Class 2"]).to_csv(OUTPUT_DIR / "confusion_matrix.csv")

misclassified = pd.DataFrame({
    "sample_no": np.arange(1, len(testing_set) + 1)[mis_mask],
    "true_class": true_labels[mis_mask],
    "predicted_class": predictions[mis_mask],
    "x1": testing_set.loc[mis_mask, "x1"].to_numpy(),
    "x2": testing_set.loc[mis_mask, "x2"].to_numpy(),
    "output_node_1": raw_outputs[mis_mask, 0],
    "output_node_2": raw_outputs[mis_mask, 1],
})
misclassified.to_csv(OUTPUT_DIR / "misclassified_samples.csv", index=False)

pd.DataFrame([{
    "samples_tested": len(testing_set),
    "class_1_true": int(np.sum(true_labels == 1)),
    "class_2_true": int(np.sum(true_labels == 2)),
    "correct_predictions": int(np.sum(predictions == true_labels)),
    "misclassified_samples": int(np.sum(mis_mask)),
    "accuracy": accuracy,
    "final_training_mse": float(history.iloc[-1]["mean_squared_error"]),
}]).to_csv(OUTPUT_DIR / "model_performance_results.csv", index=False)

history.to_csv(OUTPUT_DIR / "training_error_history.csv", index=False)
pd.DataFrame({
    "sample_no": np.arange(1, len(testing_set) + 1),
    "true_class": true_labels,
    "predicted_class": predictions,
    "output_node_1": raw_outputs[:, 0],
    "output_node_2": raw_outputs[:, 1],
}).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

# Required confusion-matrix image.
fig, ax = plt.subplots(figsize=(6, 5), dpi=220)
image = ax.imshow(confusion)
ax.set_xticks([0, 1], labels=["Class 1", "Class 2"])
ax.set_yticks([0, 1], labels=["Class 1", "Class 2"])
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title("Confusion Matrix")
threshold = float(confusion.max()) / 2.0 if confusion.size and confusion.max() else 0.0
for row in range(2):
    for col in range(2):
        ax.text(col, row, str(confusion[row, col]), ha="center", va="center", color="white" if confusion[row, col] > threshold else "black")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", bbox_inches="tight")
plt.close(fig)

# Diagnostic training-error plot.
fig, ax = plt.subplots(figsize=(7, 5), dpi=220)
ax.plot(history["epoch"], history["mean_squared_error"])
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean squared error")
ax.set_title("Training Error Across 500 Epochs")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_error_curve.png", bbox_inches="tight")
plt.close(fig)

# Display the required results directly in the notebook.
display(pd.read_csv(OUTPUT_DIR / "model_performance_results.csv"))
display(pd.read_csv(OUTPUT_DIR / "confusion_matrix.csv"))
display(pd.read_csv(OUTPUT_DIR / "misclassified_samples.csv").head(20))


,samples_tested,class_1_true,class_2_true,correct_predictions,misclassified_samples,accuracy,final_training_mse
0,1000,500,500,932,68,0.932,0.045296


,Unnamed: 0,Predicted Class 1,Predicted Class 2
0,Actual Class 1,465,35
1,Actual Class 2,33,467


,sample_no,true_class,predicted_class,x1,x2,output_node_1,output_node_2
0,11,1,2,0.514235,0.511510,0.495861,0.504042
1,16,1,2,0.646909,0.359998,0.046271,0.954208
2,25,1,2,0.512555,0.389556,0.446785,0.552577
3,37,1,2,0.600282,0.534144,0.085117,0.915069
4,66,1,2,0.566537,0.402272,0.136361,0.863409
5,94,1,2,0.565492,0.461995,0.149549,0.850244
6,98,1,2,0.525036,0.461952,0.375274,0.624270
7,120,1,2,0.546774,0.494012,0.237581,0.762057
8,133,1,2,0.542791,0.626555,0.312695,0.687261
9,134,1,2,0.525676,0.614193,0.446737,0.553412


In [8]:
def create_package(output_dir: Path, package_path: Path) -> Path:
    """Collect generated artifacts and create a single downloadable ZIP archive."""
    import shutil
    import zipfile

    # Include the supplied original workbook when it is available in the working directory.
    original_xls = Path("Computer_Assignment_4_Data.xls")
    if original_xls.exists():
        shutil.copy2(original_xls, output_dir / original_xls.name)

    # Include reproducibility/documentation files when running from the delivered project folder.
    companion_files = [
        "Amirmohsen_Sharifi_HW4_MLP_BP.py",
        "Amirmohsen_Sharifi_HW4_MLP_BP.ipynb",
        "CHANGELOG_AUDIT_REPORT.md",
    ]
    for filename in companion_files:
        source = Path(filename)
        destination = output_dir / filename
        if source.exists() and source.resolve() != destination.resolve():
            shutil.copy2(source, destination)

    with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in sorted(output_dir.iterdir()):
            if file_path.is_file() and file_path.name != package_path.name:
                archive.write(file_path, arcname=file_path.name)

    return package_path

package_path = create_package(OUTPUT_DIR, OUTPUT_DIR / "assignment_4_results.zip")
print(f"Created package: {package_path.resolve()}")

# Trigger an automatic browser download when the notebook is running in Google Colab.
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

if colab_files is not None:
    colab_files.download(str(package_path))


Created package: /content/assignment_4_results/assignment_4_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>